# EdgeQuake — AI Early-Magnitude Model v2 (distance-conditioned)

**Task**: single-station, first seconds of P (4 s window) + estimated hypocentral distance (with train-time jitter simulating live location error; v1 without distance stopped at test MAE 0.59 with heavy shrinkage on M5+) → **final catalog
magnitude** with uncertainty (Gaussian head: μ ± σ).

**Protocol (honest temporal split)**
- train/dev: **2019** (dev split is **by event**, never by trace — all
  stations of one event stay on the same side)
- test: **2020 + 2021** (never touched during training)
- blind test (later, on laptop): 2024-04-03 Hualien M7.2 & 2025-01-21 Dapu
  ML6.4 from raw GDMS data

**How to run**: attach the `magwin` Dataset (the `data/magwin` folder made by
`scripts/extract_magwin.py`), enable GPU, *Run All* (~30 min on T4).
Download afterwards: `magnet_cwa.pt`, `metrics.json`, `history.json`,
`magnet_eval.png` → put into the repo's `outputs/` folder.

Known science caveat we EXPECT to see: magnitude saturation — the first 3 s
of P underestimates M≳6.5 events. That is a documented limitation of
early-magnitude estimation, not a bug.

In [ ]:
import glob, hashlib, json, os, time
import numpy as np
import torch
import torch.nn as nn

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEV)

FILES = sorted(glob.glob('/kaggle/input/**/magwin_*.npz', recursive=True))
print('found:', FILES)
assert FILES, 'attach the magwin Dataset (magwin_2019.npz etc.)'

TRAIN_YEARS = ('2019',)
TEST_YEARS  = ('2020', '2021')
PRE_N, WIN, JIT = 200, 400, 40   # stored: P @ sample 200 of 600 (100 Hz)
EPOCHS, BS, LR = 24, 512, 1e-3
OUT = '/kaggle/working'

In [ ]:
def load_years(years):
    Xs, As, ys, Es, Ds = [], [], [], [], []
    for f in FILES:
        year = f.rsplit('_', 1)[-1].replace('.npz', '')
        if year not in years:
            continue
        d = np.load(f)
        assert 'dist' in d, ('npz has no dist column — re-run the v2 scripts/extract_magwin.py')
        Xs.append(d['X']); As.append(d['amp'])
        ys.append(d['y']); Es.append(d['event'])
        Ds.append(d['dist'])
        print(f'  {f}: {len(d["y"])} windows, {len(set(d["event"]))} events')
    return (np.concatenate(Xs), np.concatenate(As),
            np.concatenate(ys), np.concatenate(Es),
            np.concatenate(Ds))

print('train years:'); Xtr, Atr, ytr, Etr, Dtr = load_years(TRAIN_YEARS)
print('test years:');  Xte, Ate, yte, Ete, Dte = load_years(TEST_YEARS)

# dev split BY EVENT (10%): trace-level splits leak — every event is seen
# by many stations, and shape features are event-correlated
ev_hash = np.array([int(hashlib.md5(e.encode()).hexdigest()[:8], 16) % 10
                    for e in Etr])
is_dev = ev_hash == 0
print(f'train {(~is_dev).sum()} / dev {is_dev.sum()} windows | '
      f'train events {len(set(Etr[~is_dev]))} / dev events {len(set(Etr[is_dev]))}')
assert not (set(Etr[~is_dev]) & set(Etr[is_dev])), 'event leakage!'

# per-sample loss weights from magnitude-bin frequency (large events are
# rare; unweighted training collapses to the M3 mode)
bins = np.round(ytr * 2) / 2
uniq, cnt = np.unique(bins, return_counts=True)
freq = dict(zip(uniq, cnt))
w_all = np.array([(1.0 / freq[b]) ** 0.5 for b in bins], dtype=np.float32)
w_all = np.clip(w_all / w_all.mean(), 0.2, 20.0)

In [ ]:
class MagWin(torch.utils.data.Dataset):
    def __init__(self, X, A, y, D, w=None, train=False):
        self.X, self.A, self.y, self.D = X, A, y, D
        self.w = w if w is not None else np.ones(len(y), dtype=np.float32)
        self.train = train

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        x = self.X[i].astype(np.float32)          # (3, 600), peak-normalized
        j = np.random.randint(-JIT, JIT + 1) if self.train else 0
        s = (PRE_N - 100) + j                      # P lands at sample ~100
        c = x[:, s:s + WIN]
        peak0 = 10.0 ** self.A[i, 0]               # original counts scale
        pc = float(np.abs(c).max()) + 1e-12
        d = float(self.D[i])
        if not np.isfinite(d) or d <= 0:
            d = 40.0                      # fallback prior
        if self.train:                    # +-20% location error
            d *= 10.0 ** np.random.uniform(-0.08, 0.08)
        aux = np.array([np.log10(pc * peak0),
                        np.log10(float(c.std()) * peak0 + 1e-12),
                        np.log10(max(d, 1.0))],
                       dtype=np.float32)
        return c / pc, aux, self.y[i], self.w[i]


dl_tr = torch.utils.data.DataLoader(
    MagWin(Xtr[~is_dev], Atr[~is_dev], ytr[~is_dev], Dtr[~is_dev],
           w_all[~is_dev], train=True),
    batch_size=BS, shuffle=True, num_workers=2, drop_last=True)
dl_dev = torch.utils.data.DataLoader(
    MagWin(Xtr[is_dev], Atr[is_dev], ytr[is_dev], Dtr[is_dev]),
    batch_size=1024, num_workers=2)
dl_te = torch.utils.data.DataLoader(
    MagWin(Xte, Ate, yte, Dte), batch_size=1024, num_workers=2)

In [ ]:
# KEEP IN SYNC with src/edgequake/models/magnet.py in the repo
class MagNet(nn.Module):
    N_IN = 400

    def __init__(self, n_aux=2, dropout=0.2):
        super().__init__()
        def blk(ci, co, k, s):
            return [nn.Conv1d(ci, co, k, stride=s, padding=k // 2),
                    nn.BatchNorm1d(co), nn.ReLU()]
        self.conv = nn.Sequential(
            *blk(3, 32, 7, 2), *blk(32, 64, 5, 2),
            *blk(64, 64, 3, 2), *blk(64, 128, 3, 2))
        self.head = nn.Sequential(
            nn.Linear(256 + n_aux, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 2))

    def forward(self, x, aux):
        h = self.conv(x)
        h = torch.cat([h.mean(dim=2), h.amax(dim=2)], dim=1)
        out = self.head(torch.cat([h, aux], dim=1))
        return out[:, 0], out[:, 1].clamp(-6.0, 3.0)


def gaussian_nll(mu, log_var, y, weight=None):
    nll = 0.5 * (log_var + (y - mu) ** 2 / torch.exp(log_var))
    if weight is not None:
        nll = nll * weight
    return nll.mean()


model = MagNet(n_aux=3).to(DEV)
n_par = sum(p.numel() for p in model.parameters())
print(f'MagNet parameters: {n_par/1e3:.0f}k')

In [ ]:
@torch.no_grad()
def predict(loader):
    model.eval()
    mus, sigs, ys = [], [], []
    for c, aux, y, _ in loader:
        mu, lv = model(c.to(DEV), aux.to(DEV))
        mus.append(mu.cpu()); sigs.append(torch.exp(0.5 * lv).cpu())
        ys.append(y)
    return (torch.cat(mus).numpy(), torch.cat(sigs).numpy(),
            torch.cat(ys).numpy())


opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt, T_max=EPOCHS * len(dl_tr), eta_min=1e-5)

# pre-flight sanity: one forward pass must be finite and sane
c0, a0, y0, w0 = next(iter(dl_tr))
mu0, lv0 = model(c0.to(DEV), a0.to(DEV))
l0 = gaussian_nll(mu0, lv0, y0.to(DEV).float(), w0.to(DEV))
print('sanity initial loss:', float(l0))
assert torch.isfinite(l0), 'non-finite initial loss'

history, best_mae = [], 9e9
for ep in range(1, EPOCHS + 1):
    model.train()
    t0, tot, nb = time.time(), 0.0, 0
    for c, aux, y, w in dl_tr:
        opt.zero_grad(set_to_none=True)
        mu, lv = model(c.to(DEV), aux.to(DEV))
        loss = gaussian_nll(mu, lv, y.to(DEV).float(), w.to(DEV))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step(); sched.step()
        tot += float(loss); nb += 1
    mu_d, sig_d, y_d = predict(dl_dev)
    mae = float(np.abs(mu_d - y_d).mean())
    history.append({'epoch': ep, 'train_nll': tot / nb, 'dev_mae': mae})
    star = ''
    if mae < best_mae:
        best_mae = mae
        torch.save(model.state_dict(), f'{OUT}/magnet_cwa_v2.pt')
        star = '  <- saved'
    print(f'ep {ep:02d}  nll {tot/nb:.3f}  dev MAE {mae:.3f}  '
          f'({time.time()-t0:.0f}s){star}')

json.dump(history, open(f'{OUT}/history.json', 'w'), indent=1)
assert best_mae < 1.0, f'dev MAE {best_mae:.2f} looks broken — investigate'

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

model.load_state_dict(torch.load(f'{OUT}/magnet_cwa_v2.pt', map_location=DEV))
mu_t, sig_t, y_t = predict(dl_te)
err = mu_t - y_t

BINS = [(0, 3), (3, 4), (4, 5), (5, 6), (6, 9)]
per_bin = {}
for lo, hi in BINS:
    m = (y_t >= lo) & (y_t < hi)
    if m.sum():
        per_bin[f'{lo}-{hi}'] = dict(
            n=int(m.sum()), mae=float(np.abs(err[m]).mean()),
            bias=float(err[m].mean()))

z = err / sig_t
cov1 = float((np.abs(z) <= 1).mean())
cov2 = float((np.abs(z) <= 2).mean())
metrics = dict(
    n_test=len(y_t), test_years=list(TEST_YEARS),
    mae=float(np.abs(err).mean()), bias=float(err.mean()),
    per_bin=per_bin, coverage_1sigma=cov1, coverage_2sigma=cov2,
    dev_best_mae=best_mae, n_params=n_par)
json.dump(metrics, open(f'{OUT}/metrics.json', 'w'), indent=1)
print(json.dumps(metrics, indent=1))

fig, ax = plt.subplots(2, 2, figsize=(11, 9))
fig.patch.set_facecolor('#1a1a19')
for a in ax.flat:
    a.set_facecolor('#20201f')
    for s in a.spines.values(): s.set_color('#4a4a48')
    a.tick_params(colors='#c3c2b7', labelsize=8)
    a.xaxis.label.set_color('#c3c2b7'); a.yaxis.label.set_color('#c3c2b7')
    a.title.set_color('white')

hb = ax[0, 0].hexbin(y_t, mu_t, gridsize=45, cmap='cividis', mincnt=1)
ax[0, 0].plot([2, 7.5], [2, 7.5], color='#d95926', lw=1, ls='--')
ax[0, 0].set(xlabel='catalog M', ylabel='predicted M',
             title=f'test 2020-21 (single station, P+3s) MAE {metrics["mae"]:.2f}')

names = list(per_bin); maes = [per_bin[k]['mae'] for k in names]
ax[0, 1].bar(names, maes, color='#3987e5')
ax[0, 1].set(xlabel='magnitude bin', ylabel='MAE', title='MAE per bin')

ax[1, 0].hist(np.clip(z, -5, 5), bins=60, density=True, color='#3987e5')
xs = np.linspace(-5, 5, 200)
ax[1, 0].plot(xs, np.exp(-xs**2 / 2) / np.sqrt(2 * np.pi),
              color='#d95926', lw=1.2)
ax[1, 0].set(xlabel='z = (mu - y) / sigma', title=(
    f'calibration: |z|<1: {cov1:.0%} (ideal 68%), |z|<2: {cov2:.0%} (95%)'))

order = np.argsort(sig_t)
qs = np.array_split(order, 12)
ax[1, 1].plot([sig_t[q].mean() for q in qs],
              [np.abs(err[q]).mean() for q in qs],
              'o-', color='#3987e5', ms=4)
lim = max(sig_t.mean() * 3, 1)
ax[1, 1].plot([0, lim], [0, lim * 0.8], ls='--', color='#4a4a48', lw=0.8)
ax[1, 1].set(xlabel='predicted sigma', ylabel='actual |error|',
             title='does sigma track real error?')

plt.tight_layout()
plt.savefig(f'{OUT}/magnet_eval.png', dpi=140,
            facecolor=fig.get_facecolor())
print('wrote magnet_eval.png')

## Done — download these into the repo's `outputs/` folder

- `magnet_cwa.pt` — model weights (used by the live engine & blind test)
- `metrics.json`, `history.json`, `magnet_eval.png`

Next on the laptop: blind test on 0403/Dapu GDMS raw waveforms
(`scripts/blind_magnet.py`, coming with the engine integration).